## PHASE 0: PREPARATION/SETUP

### IMPORTING LIBRARIES

In [1]:
import requests
from bs4 import BeautifulSoup
import os
import pandas as pd
from sqlalchemy import create_engine, text
import re
from db_engine import my_engine

### MYSQL CONNECTION ENGINE

## PHASE 1: SCRAPING/FETCHING

### SCRAPER FUNCTION

In [2]:
engine = my_engine()

In [3]:
samp_table = """SELECT distinct a.* 
                FROM loan_applications a
                where appno in (61, 62, 63)"""
samp_df = pd.read_sql(samp_table, engine)
samp_df

,appno,applicant_name,amount,term_months,interest_rate,status,created_date,updated_date,remarks,branch
0,61,Applicant 1,10000.0,12,5.5,Pending,2025-01-01,2025-01-01,N/A,Iloilo
1,62,Applicant 2,15000.0,18,6.0,Approved,2025-01-02,2025-01-02,Urgent,Cebu
2,63,Applicant 3,20000.0,24,5.8,Rejected,2025-01-03,2025-01-03,Incomplete Docs,Manila


In [5]:
def get_dataset():
    data_url = 'https://catalog.data.gov/dataset/'
    #request
    data_website = requests.get(data_url).text
    data_soup = BeautifulSoup(data_website, 'html.parser')
    
    #find the href the main dataset 
    cons_url = 'https://catalog.data.gov'
    data_name = data_soup.find('h3', {'class' : 'dataset-heading'})
    a_tag = data_name.find('a')
    if a_tag:
        title = cons_url + a_tag['href']

    #download the dataset
    dataset_website = requests.get(title).text
    dataset_soup = BeautifulSoup(dataset_website, 'html.parser')
    
    #get the json file
    json_file = dataset_soup.find_all('a', {'class' : 'btn btn-primary'})
    link_list = []
    for href in json_file:
        links = href['href']
        link_list.append(links)
    
    #saving the official json link to download
    official_jsonlink = link_list[0]
    
    #path for saving
    csv_path = r'D:\desktop\Projects\PROJECT_7_ETL_WITH_API\TASK1_PYTHON_SCRAPER\FILES\samp_csv.csv'

    response = requests.get(official_jsonlink)

    #save the json file
    with open(csv_path, 'w', encoding='utf-8') as f:
        f.write(response.text)

#import the dataset
def import_data(path):
    for file in os.listdir(path):
        import_csv = os.path.join(path,file)
        csv = pd.read_csv(import_csv)
    return csv

In [6]:
#scrape the dataset
get_dataset()

#import the dataset
csv_path = r'D:\desktop\PROJECTS\PROJECT_1_ETL\TASK1_PYTHON_SCRAPER\FILES'
dataset = import_data(csv_path)

## PHASE 2: CLEANING AND TRANSFORMATION

In [ ]:

#rename the column country
dataset = dataset.rename(columns={'County' : 'Country'})

#clean the columns
clean_column_copy = dataset.columns
transformed_col_list = []
for col in clean_column_copy:
    cleaned_col = re.sub(r'[0-9]','', col)
    cleaned_col = re.sub(r'\(.*?\)', '', cleaned_col).strip()
    cleaned_col = re.sub(r'\s+', '_', cleaned_col)
    cleaned_col = cleaned_col.upper()
    transformed_col_list.append(cleaned_col)

#assigng new column names to dataframe
dataset.columns = transformed_col_list

#convert all the data to uppercase
dataset = dataset.map(lambda x: x.upper() if isinstance(x, str) else x)

#handling missing values
dataset = dataset.fillna("")

## PHASE 3: LOADING INTO MySQL

In [98]:
#function to create table automatically
def create_table(df, table_name):
    #save column names to list
    dataset_columns = df.columns

    #create table script and table name
    sql_create_name_beg = f'CREATE TABLE {table_name} (\n'

    #datatype
    sql_varchar = 'VARCHAR(500)'
    
    #columns with data types
    column_defs = []

    #loop the column list
    for i in dataset_columns:
        col_names = f'{i.strip()} {sql_varchar}'
        column_defs.append(col_names)

    #join the column names with comma
    sql_columns = ",\n".join(column_defs)

    #concat all to make create table script
    create_sql = f"""{sql_create_name_beg}{sql_columns});
    """
    return create_sql.strip()

In [105]:
#create table automatically in mysql
table_name = 'kjcs_project_7_tab'
create_table_script = create_table(dataset, table_name)

#connect to database and create table
with engine.connect() as conn:
    conn.execute(text(create_table_script))
    conn.commit()

In [107]:
#upload the dataset to created table
with engine.connect() as conn:
    dataset.to_sql(
        name = 'kjcs_project_7_tab',
        con = conn,
        if_exists = 'append',
        index = False
    )

In [108]:
#check if the sql exist
query = pd.read_sql('SELECT * FROM kjcs_project_7_tab LIMIT 1', engine)
query

,VIN,COUNTRY,CITY,STATE,POSTAL_CODE,MODEL_YEAR,MAKE,MODEL,ELECTRIC_VEHICLE_TYPE,CLEAN_ALTERNATIVE_FUEL_VEHICLE_ELIGIBILITY,ELECTRIC_RANGE,BASE_MSRP,LEGISLATIVE_DISTRICT,DOL_VEHICLE_ID,VEHICLE_LOCATION,ELECTRIC_UTILITY,CENSUS_TRACT
0,WA1E2AFY8R,THURSTON,OLYMPIA,WA,98512,2024,AUDI,Q5 E,PLUG-IN HYBRID ELECTRIC VEHICLE (PHEV),NOT ELIGIBLE DUE TO LOW BATTERY RANGE,23,0,22,263239938,POINT (-122.90787 46.9461),PUGET SOUND ENERGY INC,53067010910


## PHASE 4: API LAYER WITH FastAPI

#### TO DO:

1. Encrypt the credentials
2. Load the data to PowerBi